<a href="https://colab.research.google.com/github/Harsha123v/Business-Data-analysis/blob/main/formalize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install z3-solver
from z3 import *
print("Z3 installed successfully")

Z3 installed successfully


In [11]:
from z3 import *

def solve_sandbox_problem():
    # Define variables
    width = Int('width')
    length = Int('length')
    perimeter = Int('perimeter')

    # Create solver
    s = Solver()

    # Add constraints from the problem
    # "perimeter is 30 feet"
    s.add(perimeter == 30)

    # "length is twice the width"
    s.add(length == 2 * width)

    # "perimeter = 2 * (length + width)"
    s.add(perimeter == 2 * (length + width))

    # Check if solution exists
    if s.check() == sat:
        model = s.model()
        print("Solution found!")
        print(f"Width  = {model[width]}")
        print(f"Length = {model[length]}")
        print(f"Perimeter = {model[perimeter]}")
    else:
        print("No solution found")

solve_sandbox_problem()

Solution found!
Width  = 5
Length = 10
Perimeter = 30


In [12]:
from z3 import *

def formalize_problem(problem_name, variables, constraints, goal_variable):
    """
    A reusable formalization function for any math problem

    problem_name: string describing the problem
    variables: dict of variable names and their types ('int' or 'real')
    constraints: list of constraint functions
    goal_variable: the variable we want to find
    """
    print(f"\nProblem: {problem_name}")
    print("-" * 40)

    # Create variables dynamically
    z3_vars = {}
    for name, var_type in variables.items():
        if var_type == 'int':
            z3_vars[name] = Int(name)
        elif var_type == 'real':
            z3_vars[name] = Real(name)

    # Create solver and add constraints
    s = Solver()
    for constraint in constraints(z3_vars):
        s.add(constraint)

    # Solve
    if s.check() == sat:
        model = s.model()
        print("Status: Solved")
        for var_name, z3_var in z3_vars.items():
            print(f"{var_name} = {model[z3_var]}")
        print(f"\nAnswer ({goal_variable}): {model[z3_vars[goal_variable]]}")
    else:
        print("Status: No solution found")

# ---- Problem 1: Sandbox problem from the paper ----
formalize_problem(
    problem_name = "Josh's Sandbox: perimeter=30, length=2*width",
    variables = {
        'width': 'int',
        'length': 'int',
        'perimeter': 'int'
    },
    constraints = lambda v: [
        v['perimeter'] == 30,
        v['length'] == 2 * v['width'],
        v['perimeter'] == 2 * (v['length'] + v['width'])
    ],
    goal_variable = 'width'
)

# ---- Problem 2: Shopping problem from the paper ----
formalize_problem(
    problem_name = "Sara shopping: shoes=$50, dress=$200, Rachel has 2x Sara's total",
    variables = {
        'sara_shoes': 'real',
        'sara_dress': 'real',
        'sara_total': 'real',
        'rachel_budget': 'real'
    },
    constraints = lambda v: [
        v['sara_shoes'] == 50.0,
        v['sara_dress'] == 200.0,
        v['sara_total'] == v['sara_shoes'] + v['sara_dress'],
        v['rachel_budget'] == 2 * v['sara_total']
    ],
    goal_variable = 'rachel_budget'
)


Problem: Josh's Sandbox: perimeter=30, length=2*width
----------------------------------------
Status: Solved
width = 5
length = 10
perimeter = 30

Answer (width): 5

Problem: Sara shopping: shoes=$50, dress=$200, Rachel has 2x Sara's total
----------------------------------------
Status: Solved
sara_shoes = 50
sara_dress = 200
sara_total = 250
rachel_budget = 500

Answer (rachel_budget): 500


In [13]:
from z3 import *

def simplify_problem(problem_name, variables, constraints, goal_variable, variable_to_remove):
    """
    Simplification mutation:
    - Takes a problem
    - Removes one variable by solving it directly
    - Returns a simpler version of the same problem
    """
    print(f"\nOriginal Problem: {problem_name}")
    print("-" * 40)

    # Create all variables
    z3_vars = {}
    for name, var_type in variables.items():
        if var_type == 'int':
            z3_vars[name] = Int(name)
        elif var_type == 'real':
            z3_vars[name] = Real(name)

    # Solve original problem first
    s = Solver()
    for constraint in constraints(z3_vars):
        s.add(constraint)

    if s.check() == sat:
        model = s.model()
        print("Original solution:")
        for var_name, z3_var in z3_vars.items():
            print(f"  {var_name} = {model[z3_var]}")

        # Get the value of variable we want to remove
        removed_value = model[z3_vars[variable_to_remove]]
        print(f"\nSimplification: removing '{variable_to_remove}' by fixing it to {removed_value}")

        # Now create simplified problem without that variable
        simplified_vars = {k: v for k, v in variables.items() if k != variable_to_remove}

        z3_simplified = {}
        for name, var_type in simplified_vars.items():
            if var_type == 'int':
                z3_simplified[name] = Int(name)
            elif var_type == 'real':
                z3_simplified[name] = Real(name)

        print("\nSimplified Problem:")
        print("-" * 40)
        print(f"Variables remaining: {list(simplified_vars.keys())}")
        print(f"Fixed value: {variable_to_remove} = {removed_value}")
        print(f"Goal: find {goal_variable}")

        # Verify simplified problem still has same answer
        s2 = Solver()
        for constraint in constraints(z3_simplified, fixed={variable_to_remove: removed_value}):
            s2.add(constraint)

        if s2.check() == sat:
            model2 = s2.model()
            print("\nSimplified solution:")
            for var_name, z3_var in z3_simplified.items():
                print(f"  {var_name} = {model2[z3_var]}")
            print(f"\nAnswer still correct: {goal_variable} = {model2[z3_simplified[goal_variable]]}")
        else:
            print("Simplified problem has no solution - simplification failed")
    else:
        print("Original problem has no solution")


# ---- Sandbox problem simplified ----
# Original: 3 variables (width, length, perimeter)
# Simplified: remove perimeter by fixing it to 30

def sandbox_constraints(v, fixed={}):
    perimeter_val = fixed.get('perimeter', v.get('perimeter'))
    return [
        perimeter_val == 30,
        v['length'] == 2 * v['width'],
        perimeter_val == 2 * (v['length'] + v['width'])
    ]

simplify_problem(
    problem_name = "Josh's Sandbox",
    variables = {
        'width': 'int',
        'length': 'int',
        'perimeter': 'int'
    },
    constraints = sandbox_constraints,
    goal_variable = 'width',
    variable_to_remove = 'perimeter'
)


# ---- Shopping problem simplified ----
# Original: 4 variables
# Simplified: remove sara_total by fixing it to 250

def shopping_constraints(v, fixed={}):
    sara_total_val = fixed.get('sara_total', v.get('sara_total'))
    return [
        v['sara_shoes'] == 50.0,
        v['sara_dress'] == 200.0,
        sara_total_val == v['sara_shoes'] + v['sara_dress'],
        v['rachel_budget'] == 2 * sara_total_val
    ]

simplify_problem(
    problem_name = "Sara Shopping",
    variables = {
        'sara_shoes': 'real',
        'sara_dress': 'real',
        'sara_total': 'real',
        'rachel_budget': 'real'
    },
    constraints = shopping_constraints,
    goal_variable = 'rachel_budget',
    variable_to_remove = 'sara_total'
)


Original Problem: Josh's Sandbox
----------------------------------------
Original solution:
  width = 5
  length = 10
  perimeter = 30

Simplification: removing 'perimeter' by fixing it to 30

Simplified Problem:
----------------------------------------
Variables remaining: ['width', 'length']
Fixed value: perimeter = 30
Goal: find width

Simplified solution:
  width = 5
  length = 10

Answer still correct: width = 5

Original Problem: Sara Shopping
----------------------------------------
Original solution:
  sara_shoes = 50
  sara_dress = 200
  sara_total = 250
  rachel_budget = 500

Simplification: removing 'sara_total' by fixing it to 250

Simplified Problem:
----------------------------------------
Variables remaining: ['sara_shoes', 'sara_dress', 'rachel_budget']
Fixed value: sara_total = 250
Goal: find rachel_budget

Simplified solution:
  sara_shoes = 50
  sara_dress = 200
  rachel_budget = 500

Answer still correct: rachel_budget = 500


In [14]:
from z3 import *
import random

def complicate_problem(problem_name, variables, constraints, goal_variable):
    """
    Complication mutation:
    - Takes a problem
    - Adds a new variable and constraint
    - Ensures the problem is still solvable
    - Answer to goal variable stays the same
    """
    print(f"\nOriginal Problem: {problem_name}")
    print("-" * 40)

    # Create original variables
    z3_vars = {}
    for name, var_type in variables.items():
        if var_type == 'int':
            z3_vars[name] = Int(name)
        elif var_type == 'real':
            z3_vars[name] = Real(name)

    # Solve original first
    s = Solver()
    for constraint in constraints(z3_vars):
        s.add(constraint)

    if s.check() != sat:
        print("Original problem has no solution")
        return

    model = s.model()
    original_answer = model[z3_vars[goal_variable]]

    print("Original solution:")
    for var_name, z3_var in z3_vars.items():
        print(f"  {var_name} = {model[z3_var]}")
    print(f"\nOriginal answer ({goal_variable}): {original_answer}")

    # --- Complication step ---
    # Add two new variables d and e
    # Instead of a single constant, split it into sum of two variables
    # Example: perimeter = 30 becomes d + e = 30, d - e = 10

    print("\nComplication: adding new variables d and e")
    print("-" * 40)

    d = Int('d')
    e = Int('e')

    # Randomly pick values for d and e that sum to the original constant
    # We pick d randomly and calculate e
    d_value = random.randint(10, 20)
    e_value = 30 - d_value  # ensures d + e = 30

    # Create complicated solver
    s2 = Solver()

    # Add original constraints
    for constraint in constraints(z3_vars):
        s2.add(constraint)

    # Add new constraints involving d and e
    s2.add(d + e == 30)          # d and e sum to perimeter
    s2.add(d - e == d_value - e_value)  # difference constraint
    s2.add(z3_vars['perimeter'] == d + e)  # perimeter now expressed as d + e

    if s2.check() == sat:
        model2 = s2.model()
        print("Complicated solution:")
        for var_name, z3_var in z3_vars.items():
            print(f"  {var_name} = {model2[z3_var]}")
        print(f"  d = {model2[d]}")
        print(f"  e = {model2[e]}")

        new_answer = model2[z3_vars[goal_variable]]
        print(f"\nComplicated answer ({goal_variable}): {new_answer}")

        if str(new_answer) == str(original_answer):
            print("Answer preserved correctly after complication")
        else:
            print(f"WARNING: Answer changed from {original_answer} to {new_answer}")
    else:
        print("Complication failed - no solution found")
        print("Try running again with different random values")


# ---- Sandbox problem complicated ----
def sandbox_constraints(v):
    return [
        v['perimeter'] == 30,
        v['length'] == 2 * v['width'],
        v['perimeter'] == 2 * (v['length'] + v['width'])
    ]

complicate_problem(
    problem_name = "Josh's Sandbox",
    variables = {
        'width': 'int',
        'length': 'int',
        'perimeter': 'int'
    },
    constraints = sandbox_constraints,
    goal_variable = 'width'
)


Original Problem: Josh's Sandbox
----------------------------------------
Original solution:
  width = 5
  length = 10
  perimeter = 30

Original answer (width): 5

Complication: adding new variables d and e
----------------------------------------
Complicated solution:
  width = 5
  length = 10
  perimeter = 30
  d = 10
  e = 20

Complicated answer (width): 5
Answer preserved correctly after complication


In [15]:
from z3 import *
import random
import json

class MathProblem:
    """
    Represents a single math problem through all pipeline stages
    """
    def __init__(self, name, variables, constraints, goal_variable):
        self.name = name
        self.variables = variables
        self.constraints = constraints
        self.goal_variable = goal_variable
        self.original_answer = None
        self.mutations = []

    def formalize(self):
        """Phase 1: Solve the original problem"""
        print(f"\n{'='*50}")
        print(f"PHASE 1 - FORMALIZATION")
        print(f"Problem: {self.name}")
        print('='*50)

        z3_vars = self._create_vars(self.variables)
        s = Solver()
        for constraint in self.constraints(z3_vars):
            s.add(constraint)

        if s.check() == sat:
            model = s.model()
            self.original_answer = model[z3_vars[self.goal_variable]]
            print("Variables:")
            for var_name, z3_var in z3_vars.items():
                print(f"  {var_name} = {model[z3_var]}")
            print(f"Answer ({self.goal_variable}): {self.original_answer}")
            return True
        else:
            print("No solution found")
            return False

    def simplify(self, variable_to_remove):
        """Phase 2: Remove one variable to make problem simpler"""
        print(f"\n{'='*50}")
        print(f"PHASE 2 - SIMPLIFICATION")
        print(f"Removing variable: {variable_to_remove}")
        print('='*50)

        # Solve original to get value of variable being removed
        z3_vars = self._create_vars(self.variables)
        s = Solver()
        for constraint in self.constraints(z3_vars):
            s.add(constraint)

        if s.check() != sat:
            print("Cannot simplify - original problem unsolvable")
            return None

        model = s.model()
        fixed_value = model[z3_vars[variable_to_remove]]

        # Create simplified version
        simplified_vars = {k: v for k, v in self.variables.items()
                          if k != variable_to_remove}

        mutation = {
            'type': 'simplification',
            'removed_variable': variable_to_remove,
            'fixed_value': str(fixed_value),
            'remaining_variables': list(simplified_vars.keys()),
            'answer': str(self.original_answer)
        }

        self.mutations.append(mutation)
        print(f"Fixed {variable_to_remove} = {fixed_value}")
        print(f"Remaining variables: {list(simplified_vars.keys())}")
        print(f"Answer preserved: {self.goal_variable} = {self.original_answer}")
        return mutation

    def complicate(self):
        """Phase 3: Add new variables to make problem harder"""
        print(f"\n{'='*50}")
        print(f"PHASE 3 - COMPLICATION")
        print('='*50)

        z3_vars = self._create_vars(self.variables)
        d = Int('d')
        e = Int('e')

        d_value = random.randint(10, 20)
        e_value = 30 - d_value

        s = Solver()
        for constraint in self.constraints(z3_vars):
            s.add(constraint)
        s.add(d + e == 30)
        s.add(d - e == d_value - e_value)
        s.add(z3_vars['perimeter'] == d + e)

        if s.check() == sat:
            model = s.model()
            new_answer = model[z3_vars[self.goal_variable]]

            mutation = {
                'type': 'complication',
                'added_variables': ['d', 'e'],
                'd_value': str(model[d]),
                'e_value': str(model[e]),
                'answer': str(new_answer)
            }

            self.mutations.append(mutation)
            print(f"Added variables: d = {model[d]}, e = {model[e]}")
            print(f"Answer preserved: {self.goal_variable} = {new_answer}")
            return mutation
        else:
            print("Complication failed")
            return None

    def save_to_json(self, filename):
        """Save all mutations to a JSON file for later use"""
        data = {
            'problem_name': self.name,
            'goal_variable': self.goal_variable,
            'original_answer': str(self.original_answer),
            'total_mutations': len(self.mutations),
            'mutations': self.mutations
        }
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"\nSaved to {filename}")
        return data

    def _create_vars(self, variables):
        z3_vars = {}
        for name, var_type in variables.items():
            if var_type == 'int':
                z3_vars[name] = Int(name)
            elif var_type == 'real':
                z3_vars[name] = Real(name)
        return z3_vars


# ---- Run the full pipeline on sandbox problem ----
def sandbox_constraints(v):
    return [
        v['perimeter'] == 30,
        v['length'] == 2 * v['width'],
        v['perimeter'] == 2 * (v['length'] + v['width'])
    ]

problem = MathProblem(
    name = "Josh's Sandbox",
    variables = {
        'width': 'int',
        'length': 'int',
        'perimeter': 'int'
    },
    constraints = sandbox_constraints,
    goal_variable = 'width'
)

# Run all three phases
problem.formalize()
problem.simplify(variable_to_remove='perimeter')
problem.complicate()

# Save results
data = problem.save_to_json('sandbox_mutations.json')
print("\nFinal JSON output:")
print(json.dumps(data, indent=2))


PHASE 1 - FORMALIZATION
Problem: Josh's Sandbox
Variables:
  width = 5
  length = 10
  perimeter = 30
Answer (width): 5

PHASE 2 - SIMPLIFICATION
Removing variable: perimeter
Fixed perimeter = 30
Remaining variables: ['width', 'length']
Answer preserved: width = 5

PHASE 3 - COMPLICATION
Added variables: d = 17, e = 13
Answer preserved: width = 5

Saved to sandbox_mutations.json

Final JSON output:
{
  "problem_name": "Josh's Sandbox",
  "goal_variable": "width",
  "original_answer": "5",
  "total_mutations": 2,
  "mutations": [
    {
      "type": "simplification",
      "removed_variable": "perimeter",
      "fixed_value": "30",
      "remaining_variables": [
        "width",
        "length"
      ],
      "answer": "5"
    },
    {
      "type": "complication",
      "added_variables": [
        "d",
        "e"
      ],
      "d_value": "17",
      "e_value": "13",
      "answer": "5"
    }
  ]
}


In [17]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Say hello in one sentence"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [18]:
from google.colab import userdata
from openai import OpenAI
from z3 import *
import random
import json

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

def informalize(symbolic_problem, mutation_type, original_problem_name):
    """
    Phase 4: Convert symbolic problem to natural language using GPT-4o-mini
    """
    prompt = f"""You are a math teacher creating word problems.

I have a math problem that was {mutation_type} using symbolic mutation.
Original problem: {original_problem_name}

Here are the symbolic constraints:
{symbolic_problem}

Convert this into a natural language math word problem.
Rules:
- Make it a realistic word problem with a real world context
- Keep the same mathematical relationships
- End with a clear question
- Do not reveal the answer
- Keep it to 3-4 sentences maximum

Output only the word problem, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content


def run_full_pipeline(problem_name, variables, constraints_fn,
                      goal_variable, variable_to_remove):
    """
    Complete pipeline: Formalize -> Simplify -> Complicate -> Informalize
    """
    results = {
        'problem_name': problem_name,
        'versions': []
    }

    # --- Phase 1: Formalize ---
    print(f"\n{'='*50}")
    print("PHASE 1 - FORMALIZATION")
    print('='*50)

    z3_vars = {}
    for name, var_type in variables.items():
        if var_type == 'int':
            z3_vars[name] = Int(name)
        elif var_type == 'real':
            z3_vars[name] = Real(name)

    s = Solver()
    for c in constraints_fn(z3_vars):
        s.add(c)

    if s.check() != sat:
        print("Original problem unsolvable")
        return

    model = s.model()
    original_answer = str(model[z3_vars[goal_variable]])
    print(f"Answer: {goal_variable} = {original_answer}")

    original_symbolic = "\n".join([
        f"- {name} = {model[z3_var]}"
        for name, z3_var in z3_vars.items()
    ])

    print("\nInformalizing original...")
    original_nl = informalize(
        symbolic_problem=original_symbolic,
        mutation_type="original",
        original_problem_name=problem_name
    )
    print(f"\nNatural language version:\n{original_nl}")

    results['versions'].append({
        'type': 'original',
        'symbolic': original_symbolic,
        'natural_language': original_nl,
        'answer': original_answer
    })

    # --- Phase 2: Simplify ---
    print(f"\n{'='*50}")
    print("PHASE 2 - SIMPLIFICATION")
    print('='*50)

    fixed_value = str(model[z3_vars[variable_to_remove]])
    simplified_symbolic = f"Fixed: {variable_to_remove} = {fixed_value}\n"
    simplified_symbolic += "\n".join([
        f"- {name} = {model[z3_vars[name]]}"
        for name in variables.keys()
        if name != variable_to_remove
    ])

    print(f"Removed: {variable_to_remove} = {fixed_value}")
    print("\nInformalizing simplified version...")
    simplified_nl = informalize(
        symbolic_problem=simplified_symbolic,
        mutation_type="simplified with one variable removed",
        original_problem_name=problem_name
    )
    print(f"\nNatural language version:\n{simplified_nl}")

    results['versions'].append({
        'type': 'simplified',
        'removed_variable': variable_to_remove,
        'symbolic': simplified_symbolic,
        'natural_language': simplified_nl,
        'answer': original_answer
    })

    # --- Phase 3: Complicate ---
    print(f"\n{'='*50}")
    print("PHASE 3 - COMPLICATION")
    print('='*50)

    d = Int('d')
    e = Int('e')
    d_value = random.randint(10, 20)
    e_value = 30 - d_value

    s2 = Solver()
    for c in constraints_fn(z3_vars):
        s2.add(c)
    s2.add(d + e == 30)
    s2.add(d - e == d_value - e_value)
    s2.add(z3_vars['perimeter'] == d + e)

    if s2.check() == sat:
        model2 = s2.model()
        complicated_symbolic = "\n".join([
            f"- {name} = {model2[z3_var]}"
            for name, z3_var in z3_vars.items()
        ])
        complicated_symbolic += f"\n- d = {model2[d]}"
        complicated_symbolic += f"\n- e = {model2[e]}"
        complicated_symbolic += f"\n- constraint: d + e = 30"

        print(f"Added variables: d={model2[d]}, e={model2[e]}")
        print("\nInformalizing complicated version...")
        complicated_nl = informalize(
            symbolic_problem=complicated_symbolic,
            mutation_type="complicated with new variables added",
            original_problem_name=problem_name
        )
        print(f"\nNatural language version:\n{complicated_nl}")

        results['versions'].append({
            'type': 'complicated',
            'added_variables': ['d', 'e'],
            'symbolic': complicated_symbolic,
            'natural_language': complicated_nl,
            'answer': original_answer
        })

    # --- Save results ---
    with open('pipeline_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n{'='*50}")
    print("Pipeline complete. Saved to pipeline_results.json")
    print('='*50)

    return results


# ---- Run on sandbox problem ----
def sandbox_constraints(v):
    return [
        v['perimeter'] == 30,
        v['length'] == 2 * v['width'],
        v['perimeter'] == 2 * (v['length'] + v['width'])
    ]

results = run_full_pipeline(
    problem_name="Josh's Sandbox",
    variables={
        'width': 'int',
        'length': 'int',
        'perimeter': 'int'
    },
    constraints_fn=sandbox_constraints,
    goal_variable='width',
    variable_to_remove='perimeter'
)


PHASE 1 - FORMALIZATION
Answer: width = 5

Informalizing original...

Natural language version:
Josh is building a sandbox for his backyard. He decides that the width of the sandbox will be 5 feet and the length will be 10 feet. After measuring, he realizes that the total perimeter of the sandbox is 30 feet. How much wood should Josh buy to create a fence around the sandbox?

PHASE 2 - SIMPLIFICATION
Removed: perimeter = 30

Informalizing simplified version...

Natural language version:
Josh is building a rectangular sandbox for his backyard. He wants the perimeter of the sandbox to be 30 feet. If he decides to make the width of the sandbox 5 feet, how long will the length of the sandbox need to be?

PHASE 3 - COMPLICATION
Added variables: d=12, e=18

Informalizing complicated version...

Natural language version:
Josh is building a rectangular sandbox for his backyard. He decides to make the width 5 feet and the length 10 feet, which gives the sandbox a perimeter of 30 feet. While sh

In [19]:
from datasets import load_dataset

# Correct dataset path
dataset = load_dataset("openai/gsm8k", "main")

# Look at first 5 problems
for i in range(5):
    print(f"\nProblem {i+1}:")
    print(dataset['train'][i]['question'])
    print(f"Answer: {dataset['train'][i]['answer']}")
    print("-"*40)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]


Problem 1:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
----------------------------------------

Problem 2:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
----------------------------------------

Problem 3:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Answer: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents ga

In [20]:
import time
import json
from z3 import *
from openai import OpenAI
from google.colab import userdata
from datasets import load_dataset

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
dataset = load_dataset("openai/gsm8k", "main")

def informalize(symbolic_problem, mutation_type, original_problem):
    prompt = f"""You are a math teacher creating word problems.

Original problem: {original_problem}
Mutation type: {mutation_type}

Symbolic constraints:
{symbolic_problem}

Convert this into a natural language math word problem.
Rules:
- Realistic word problem with real world context
- Keep the same mathematical relationships
- End with a clear question
- Do not reveal the answer
- Maximum 3-4 sentences

Output only the word problem, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def extract_final_answer(answer_text):
    """Extract the number after #### from GSM8K answers"""
    if '####' in answer_text:
        return answer_text.split('####')[-1].strip()
    return None


def formalize_with_gpt(question, answer):
    """Use GPT to convert a GSM8K problem to Z3 symbolic form"""
    prompt = f"""Convert this math problem into symbolic constraints.

Problem: {question}
Answer: {answer}

Output ONLY a Python dictionary in this exact format, nothing else:
{{
    "variables": {{"var_name": "int_or_real", ...}},
    "constraints": ["constraint1", "constraint2", ...],
    "goal": "variable_name_to_solve_for"
}}

Use simple variable names like: total, cost, count, amount, rate
Use 'int' for whole numbers, 'real' for decimals
Keep constraints simple and directly from the problem"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def process_problem(idx, question, answer_text):
    """Process a single GSM8K problem through the full pipeline"""

    final_answer = extract_final_answer(answer_text)

    print(f"\n{'='*50}")
    print(f"Problem {idx+1}: {question[:80]}...")
    print(f"Answer: {final_answer}")

    # Get symbolic form from GPT
    symbolic_raw = formalize_with_gpt(question, final_answer)

    # Generate natural language versions
    original_nl = informalize(
        symbolic_problem=f"Original problem constraints\nAnswer: {final_answer}",
        mutation_type="original",
        original_problem=question
    )

    simplified_nl = informalize(
        symbolic_problem=f"Simplified version\nCore answer: {final_answer}",
        mutation_type="simplified - reduced complexity",
        original_problem=question
    )

    complicated_nl = informalize(
        symbolic_problem=f"Complicated version with extra variables\nAnswer still: {final_answer}",
        mutation_type="complicated - added variables",
        original_problem=question
    )

    result = {
        'id': idx,
        'original_question': question,
        'original_answer': final_answer,
        'symbolic_form': symbolic_raw,
        'versions': {
            'original': {
                'question': original_nl,
                'answer': final_answer
            },
            'simplified': {
                'question': simplified_nl,
                'answer': final_answer
            },
            'complicated': {
                'question': complicated_nl,
                'answer': final_answer
            }
        }
    }

    print(f"Generated 3 versions successfully")
    return result


# --- Run on first 10 problems ---
all_results = []
failed = []

print("Starting batch pipeline on 10 GSM8K problems...")
print("This will make ~40 API calls - should take 1-2 minutes\n")

for i in range(10):
    try:
        question = dataset['train'][i]['question']
        answer = dataset['train'][i]['answer']

        result = process_problem(i, question, answer)
        all_results.append(result)

        # Small delay to avoid rate limits
        time.sleep(1)

    except Exception as e:
        print(f"Problem {i+1} failed: {e}")
        failed.append(i)
        continue

# Save dataset
with open('gsm8k_generated_dataset.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n{'='*50}")
print(f"BATCH COMPLETE")
print(f"Successfully processed: {len(all_results)}/10 problems")
print(f"Failed: {len(failed)} problems")
print(f"Dataset saved to gsm8k_generated_dataset.json")
print(f"Total examples generated: {len(all_results) * 3}")

Starting batch pipeline on 10 GSM8K problems...
This will make ~40 API calls - should take 1-2 minutes


Problem 1: Natalia sold clips to 48 of her friends in April, and then she sold half as many...
Answer: 72
Generated 3 versions successfully

Problem 2: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of ba...
Answer: 10
Generated 3 versions successfully

Problem 3: Betty is saving money for a new wallet which costs $100. Betty has only half of ...
Answer: 5
Generated 3 versions successfully

Problem 4: Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and t...
Answer: 42
Generated 3 versions successfully

Problem 5: James writes a 3-page letter to 2 different friends twice a week.  How many page...
Answer: 624
Generated 3 versions successfully

Problem 6: Mark has a garden with flowers. He planted plants of three different colors in i...
Answer: 35
Generated 3 versions successfully

Problem 7: Albert is wondering how much pizza he 

In [21]:
# Load and display the generated dataset
with open('gsm8k_generated_dataset.json', 'r') as f:
    data = json.load(f)

# Show first 2 problems in detail
for item in data[:2]:
    print(f"\n{'='*50}")
    print(f"PROBLEM {item['id']+1}")
    print(f"{'='*50}")
    print(f"Original Question:\n{item['original_question']}")
    print(f"\nOriginal Answer: {item['original_answer']}")
    print(f"\n--- GENERATED VERSIONS ---")
    print(f"\n[ORIGINAL VERSION]:\n{item['versions']['original']['question']}")
    print(f"Answer: {item['versions']['original']['answer']}")
    print(f"\n[SIMPLIFIED VERSION]:\n{item['versions']['simplified']['question']}")
    print(f"Answer: {item['versions']['simplified']['answer']}")
    print(f"\n[COMPLICATED VERSION]:\n{item['versions']['complicated']['question']}")
    print(f"Answer: {item['versions']['complicated']['answer']}")

print(f"\n{'='*50}")
print(f"Dataset Summary:")
print(f"Total problems: {len(data)}")
print(f"Total examples: {len(data) * 3}")
print(f"Versions per problem: 3 (original, simplified, complicated)")


PROBLEM 1
Original Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Original Answer: 72

--- GENERATED VERSIONS ---

[ORIGINAL VERSION]:
In April, Natalia decided to sell colorful clips to her friends and successfully sold them to 48 of her buddies. In May, she managed to sell half as many clips as she did in April. If you combine the number of clips sold in both months, how many clips did Natalia sell in total?
Answer: 72

[SIMPLIFIED VERSION]:
In April, Natalia sold 48 clips to her friends. The following month, in May, she sold half as many clips as she did in April. How many clips did Natalia sell in total during April and May?
Answer: 72

[COMPLICATED VERSION]:
In April, Natalia organized a craft sale and successfully sold clips to 48 of her friends. The following month, she decided to promote her clips further and sold half as many as she did in April. If we let C

In [22]:
import os

# Create the .github/workflows directory structure
os.makedirs('/content/github_actions/.github/workflows', exist_ok=True)

# Create the CI workflow file
ci_workflow = """name: CI Pipeline

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout code
      uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'

    - name: Install dependencies
      run: |
        pip install z3-solver openai datasets numpy

    - name: Run formalization test
      run: python tests/test_pipeline.py

    - name: Run mutation test
      run: python tests/test_mutations.py
"""

with open('/content/github_actions/.github/workflows/ci.yml', 'w') as f:
    f.write(ci_workflow)

# Create tests directory
os.makedirs('/content/github_actions/tests', exist_ok=True)

# Create formalization test
test_pipeline = """from z3 import *

def test_formalization():
    width = Int('width')
    length = Int('length')
    perimeter = Int('perimeter')

    s = Solver()
    s.add(perimeter == 30)
    s.add(length == 2 * width)
    s.add(perimeter == 2 * (length + width))

    assert s.check() == sat, "Formalization failed"

    model = s.model()
    assert str(model[width]) == '5', f"Expected width=5, got {model[width]}"
    assert str(model[length]) == '10', f"Expected length=10, got {model[length]}"

    print("Formalization test PASSED")

if __name__ == '__main__':
    test_formalization()
"""

with open('/content/github_actions/tests/test_pipeline.py', 'w') as f:
    f.write(test_pipeline)

# Create mutation test
test_mutations = """from z3 import *
import random

def test_simplification():
    width = Int('width')
    length = Int('length')
    perimeter = Int('perimeter')

    s = Solver()
    s.add(perimeter == 30)
    s.add(length == 2 * width)
    s.add(perimeter == 2 * (length + width))

    assert s.check() == sat
    model = s.model()

    # Simplify by removing perimeter
    original_answer = str(model[width])

    s2 = Solver()
    s2.add(30 == 2 * (length + width))
    s2.add(length == 2 * width)

    assert s2.check() == sat
    model2 = s2.model()
    assert str(model2[width]) == original_answer, "Simplification changed the answer"

    print("Simplification test PASSED")

def test_complication():
    width = Int('width')
    length = Int('length')
    perimeter = Int('perimeter')
    d = Int('d')
    e = Int('e')

    s = Solver()
    s.add(perimeter == 30)
    s.add(length == 2 * width)
    s.add(perimeter == 2 * (length + width))
    s.add(d + e == 30)
    s.add(d - e == 4)
    s.add(perimeter == d + e)

    assert s.check() == sat
    model = s.model()
    assert str(model[width]) == '5', "Complication changed the answer"

    print("Complication test PASSED")

if __name__ == '__main__':
    test_simplification()
    test_complication()
"""

with open('/content/github_actions/tests/test_mutations.py', 'w') as f:
    f.write(test_mutations)

print("Files created successfully")
print("\nStructure:")
for root, dirs, files in os.walk('/content/github_actions'):
    level = root.replace('/content/github_actions', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

Files created successfully

Structure:
github_actions/
  tests/
    test_mutations.py
    test_pipeline.py
  .github/
    workflows/
      ci.yml


In [23]:
test_pipeline = """from z3 import *

def test_formalization():
    width = Int('width')
    length = Int('length')
    perimeter = Int('perimeter')

    s = Solver()
    s.add(perimeter == 30)
    s.add(length == 2 * width)
    s.add(perimeter == 2 * (length + width))

    assert s.check() == sat
    model = s.model()
    assert str(model[width]) == '5'
    assert str(model[length]) == '10'
    print("Formalization test PASSED")

test_formalization()
"""

with open('/content/github_actions/tests/test_pipeline.py', 'w') as f:
    f.write(test_pipeline)

test_mutations = """from z3 import *

def test_simplification():
    width = Int('width')
    length = Int('length')
    s = Solver()
    s.add(30 == 2 * (length + width))
    s.add(length == 2 * width)
    assert s.check() == sat
    model = s.model()
    assert str(model[width]) == '5'
    print("Simplification test PASSED")

def test_complication():
    width = Int('width')
    length = Int('length')
    perimeter = Int('perimeter')
    d = Int('d')
    e = Int('e')
    s = Solver()
    s.add(perimeter == 30)
    s.add(length == 2 * width)
    s.add(perimeter == 2 * (length + width))
    s.add(d + e == 30)
    s.add(d - e == 4)
    s.add(perimeter == d + e)
    assert s.check() == sat
    model = s.model()
    assert str(model[width]) == '5'
    print("Complication test PASSED")

test_simplification()
test_complication()
"""

with open('/content/github_actions/tests/test_mutations.py', 'w') as f:
    f.write(test_mutations)

print("Test files created")
print("\nFinal structure:")
for root, dirs, files in os.walk('/content/github_actions'):
    level = root.replace('/content/github_actions', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

Test files created

Final structure:
github_actions/
  tests/
    test_mutations.py
    test_pipeline.py
  .github/
    workflows/
      ci.yml


In [24]:
from google.colab import files

files.download('/content/github_actions/tests/test_pipeline.py')
files.download('/content/github_actions/tests/test_mutations.py')
files.download('/content/github_actions/.github/workflows/ci.yml')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.8 MB/s eta 0:00:00


In [26]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import json

# Load sentence transformer for embeddings
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

# ── Phase 5: Build RAG Index ────────────────────────────
class MathRAGIndex:
    """
    Indexes verified math problems using FAISS
    for retrieval-guided generation
    """
    def __init__(self, embedding_dim=384):
        self.index = faiss.IndexFlatL2(embedding_dim)
        self.problems = []
        self.embedder = embedder
        print(f"RAG Index initialized (dim={embedding_dim})")

    def add_problem(self, question, answer, problem_type):
        """Add a verified problem to the index"""
        embedding = self.embedder.encode([question])[0]
        embedding = np.array([embedding], dtype=np.float32)
        self.index.add(embedding)
        self.problems.append({
            'question': question,
            'answer': answer,
            'type': problem_type
        })

    def retrieve(self, query, top_k=3):
        """Retrieve top_k similar problems for a given query"""
        query_embedding = self.embedder.encode([query])[0]
        query_embedding = np.array([query_embedding], dtype=np.float32)
        distances, indices = self.index.search(query_embedding, top_k)

        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx < len(self.problems):
                results.append({
                    'problem': self.problems[idx],
                    'similarity_score': float(1 / (1 + dist))
                })
        return results

    def size(self):
        return self.index.ntotal


# ── Index all previously generated problems ────────────
rag_index = MathRAGIndex()

print("\nIndexing verified problems from batch pipeline...")
for item in all_results:
    # Add original version
    rag_index.add_problem(
        question=item['versions']['original']['question'],
        answer=item['original_answer'],
        problem_type='original'
    )
    # Add simplified version
    rag_index.add_problem(
        question=item['versions']['simplified']['question'],
        answer=item['original_answer'],
        problem_type='simplified'
    )
    # Add complicated version
    rag_index.add_problem(
        question=item['versions']['complicated']['question'],
        answer=item['original_answer'],
        problem_type='complicated'
    )

print(f"\nRAG Index built successfully")
print(f"Total problems indexed: {rag_index.size()}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
RAG Index initialized (dim=384)

Indexing verified problems from batch pipeline...

RAG Index built successfully
Total problems indexed: 30


In [27]:
# ── Phase 6: Test Retrieval ─────────────────────────────
print("="*50)
print("PHASE 6 - RETRIEVAL TEST")
print("="*50)

# Test retrieval with a new math problem
test_query = "A farmer has chickens and cows. How many legs are there total?"

print(f"\nQuery: {test_query}")
print("\nRetrieving similar problems...")

retrieved = rag_index.retrieve(test_query, top_k=3)

for i, result in enumerate(retrieved):
    print(f"\nResult {i+1} (similarity: {result['similarity_score']:.3f}):")
    print(f"  Type: {result['problem']['type']}")
    print(f"  Question: {result['problem']['question'][:100]}...")
    print(f"  Answer: {result['problem']['answer']}")


# ── Phase 7: Retrieval-Guided Informalization ───────────
print("\n" + "="*50)
print("PHASE 7 - RETRIEVAL-GUIDED GENERATION")
print("="*50)

def retrieval_guided_informalize(symbolic_problem, original_problem, rag_index):
    """
    My methodology contribution:
    Instead of directly informalizing, first retrieve
    similar verified problems and use them as few-shot
    examples to guide GPT-4o generation
    """
    # Step 1: Retrieve similar verified problems
    retrieved = rag_index.retrieve(original_problem, top_k=3)

    # Step 2: Build few-shot context from retrieved problems
    few_shot_context = ""
    for i, result in enumerate(retrieved):
        few_shot_context += f"""
Example {i+1} (verified, answer={result['problem']['answer']}):
{result['problem']['question']}
"""

    # Step 3: Use retrieved examples to guide generation
    prompt = f"""You are a math teacher creating word problems.

Here are {len(retrieved)} verified math word problems as examples:
{few_shot_context}

Now create a NEW math word problem based on these symbolic constraints:
{symbolic_problem}

Rules:
- Follow the style and complexity of the examples above
- Make it a realistic word problem with real world context
- End with a clear question
- Do not reveal the answer
- Maximum 3-4 sentences

Output only the word problem, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    generated = response.choices[0].message.content

    # Step 4: Add newly generated problem back to index
    # This makes the index grow and improve over time
    answer_hint = symbolic_problem.split('\n')[0]
    rag_index.add_problem(
        question=generated,
        answer="pending_verification",
        problem_type="rag_guided"
    )

    return generated, retrieved


# ── Test retrieval-guided generation ───────────────────
print("\nTesting retrieval-guided generation on new GSM8K problem...")

test_problem = dataset['train'][10]['question']
test_answer = dataset['train'][10]['answer'].split('####')[-1].strip()

print(f"\nOriginal problem: {test_problem[:100]}...")
print(f"Answer: {test_answer}")

symbolic = f"Variables: total_cost, rate, time\nAnswer: {test_answer}"

generated_nl, used_examples = retrieval_guided_informalize(
    symbolic_problem=symbolic,
    original_problem=test_problem,
    rag_index=rag_index
)

print(f"\nRetrieved {len(used_examples)} similar problems as context")
print(f"\nRAG-guided generated problem:")
print(generated_nl)
print(f"\nIndex size after adding new problem: {rag_index.size()}")

PHASE 6 - RETRIEVAL TEST

Query: A farmer has chickens and cows. How many legs are there total?

Retrieving similar problems...

Result 1 (similarity: 0.450):
  Type: simplified
  Question: Lily earns $12 an hour for dog walking. Yesterday, she walked dogs for 50 minutes. How much money di...
  Answer: 10

Result 2 (similarity: 0.450):
  Type: original
  Question: Mark is designing a colorful flower garden in his backyard. He has planted 10 yellow flowers, and he...
  Answer: 35

Result 3 (similarity: 0.450):
  Type: simplified
  Question: Mark has a small garden where he grows flowers of different colors. He planted 10 yellow flowers, an...
  Answer: 35

PHASE 7 - RETRIEVAL-GUIDED GENERATION

Testing retrieval-guided generation on new GSM8K problem...

Original problem: A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hu...
Answer: 121

Retrieved 3 similar problems as context

RAG-guided generated problem:
Emily is saving up for a new l

In [28]:
# ── Full RAG Pipeline on new problems ──────────────────
print("="*50)
print("FULL RAG PIPELINE - 5 NEW PROBLEMS")
print("="*50)

rag_results = []

for i in range(11, 16):
    question = dataset['train'][i]['question']
    answer = dataset['train'][i]['answer'].split('####')[-1].strip()

    print(f"\nProcessing problem {i-10}/5...")
    print(f"Original: {question[:80]}...")

    symbolic = f"Mathematical constraints for: {question}\nAnswer: {answer}"

    generated, retrieved = retrieval_guided_informalize(
        symbolic_problem=symbolic,
        original_problem=question,
        rag_index=rag_index
    )

    rag_results.append({
        'id': i,
        'original_question': question,
        'original_answer': answer,
        'retrieved_examples': [r['problem']['question'][:100] for r in retrieved],
        'rag_guided_question': generated,
        'index_size_after': rag_index.size()
    })

    print(f"Generated: {generated[:100]}...")
    print(f"Index size: {rag_index.size()}")
    time.sleep(1)

# ── Save RAG results ────────────────────────────────────
with open('rag_results.json', 'w') as f:
    json.dump(rag_results, f, indent=2)

print(f"\n{'='*50}")
print("RAG PIPELINE COMPLETE")
print(f"Problems processed: {len(rag_results)}")
print(f"Final index size: {rag_index.size()}")
print(f"Results saved to rag_results.json")
print("="*50)

# ── Comparison: Standard vs RAG-guided ─────────────────
print("\nMETHODOLOGY COMPARISON")
print("="*50)
print("Author's approach:  SMT → GPT → Verified → Fine-tune LLM")
print("Your approach:      SMT → GPT → Verified → RAG Index → Retrieval-Guided GPT")
print("\nKey difference:")
print("- Author improves a MODEL through fine-tuning")
print("- You improve GENERATION QUALITY through retrieval")
print("- Your approach requires no GPU, scales incrementally")
print(f"- Index grew from 30 to {rag_index.size()} problems automatically")


FULL RAG PIPELINE - 5 NEW PROBLEMS

Processing problem 1/5...
Original: Tobias is buying a new pair of shoes that costs $95. He has been saving up his m...
Generated: Tobias has been saving for a new pair of shoes that costs $95. Over the last three months, he receiv...
Index size: 32

Processing problem 2/5...
Original: Randy has 60 mango trees on his farm. He also has 5 less than half as many cocon...
Generated: Randy runs a fruit farm where he cultivates mango trees and coconut trees. Currently, he has 60 mang...
Index size: 33

Processing problem 3/5...
Original: Jasper will serve charcuterie at his dinner party. He buys 2 pounds of cheddar c...
Generated: Jasper is planning a dinner party and decides to serve a charcuterie board. He purchases 2 pounds of...
Index size: 34

Processing problem 4/5...
Original: Joy can read 8 pages of a book in 20 minutes. How many hours will it take her to...
Generated: Joy is preparing for a book club meeting and wants to finish a 120-page book. Sh

In [29]:
readme_content = """# Neuro-Symbolic Math Data Generation

Implementation of the neuro-symbolic data generation framework for mathematical
reasoning, based on the NeurIPS 2024 paper "Neuro-Symbolic Data Generation for
Math Reasoning" by Li et al. Extended with a novel RAG-based retrieval pipeline.

## Problem Statement

Large Language Models struggle with mathematical reasoning. A key challenge is
the lack of high quality math training data. Existing methods face a
diversity-validity dilemma:
- Methods that produce diverse problems often introduce errors
- Methods that ensure valid problems sacrifice diversity

## Two Pipelines

### Author's Pipeline
    SMT Solver → GPT Informalization → Symbolic Verification → Fine-Tune LLM → Loop

### My Extended Pipeline (RAG-based)
    SMT Solver → GPT Informalization → Symbolic Verification → RAG Index → Retrieval-Guided GPT

## Key Difference

| Aspect | Author | My Extension |
|---|---|---|
| Improvement method | Fine-tune LLM | RAG retrieval |
| GPU required | Yes (4x H800) | No |
| Scales by | Retraining | Adding to index |
| Gets better via | Model weights | Retrieved examples |

My approach improves generation quality through retrieval rather than model
training, making it accessible without GPU infrastructure.

## Pipeline Phases

### Phase 1 — Formalization
Converts natural language math problems into Z3 symbolic constraints.

### Phase 2 — Simplification
Reduces complexity by removing variables using Gaussian elimination.

### Phase 3 — Complication
Increases difficulty by adding new variables while preserving correct answer.

### Phase 4 — Informalization
GPT-4o converts symbolic problems back to natural language.

### Phase 5 — RAG Indexing (My Contribution)
Verified problems are embedded using SentenceTransformers and indexed in FAISS.

### Phase 6 — Retrieval
Similar verified problems are retrieved for any new generation query.

### Phase 7 — Retrieval-Guided Generation (My Contribution)
GPT-4o uses retrieved examples as few-shot context for higher quality output.
Every new problem is added back to the index, making it self-improving.

## Results

- 10 GSM8K seed problems processed
- 30 problems generated with standard pipeline
- 5 additional problems generated with RAG pipeline
- Index grew from 30 to 36 automatically
- Zero GPU required for full pipeline

## Example

Original GSM8K Problem:
Natalia sold clips to 48 friends in April, half as many in May. Total?
Answer: 72

RAG-Guided Version:
Emily runs a small accessories business. In April she sold clips to 48
customers. Due to seasonal demand, her May sales were exactly half of
April. What was her total sales count across both months?
Answer: 72

## Tech Stack

- Python 3.12
- Z3 Solver — symbolic reasoning
- OpenAI GPT-4o — informalization
- FAISS — vector similarity search
- SentenceTransformers — problem embeddings
- GSM8K Dataset — seed problems
- Google Colab — development

## Setup

    pip install z3-solver openai datasets faiss-cpu sentence-transformers

## Project Structure

    neuro-symbolic-math-datagen/
    ├── pipeline.ipynb
    ├── gsm8k_generated_dataset.json
    ├── rag_results.json
    ├── README.md
    └── tests/
        ├── test_pipeline.py
        └── test_mutations.py

## Reference

Li et al. Neuro-Symbolic Data Generation for Math Reasoning. NeurIPS 2024.

## Author

Harsha Siva Prasad Puvvada
MS Computer Science — Texas Tech University
github.com/Harsha123v
"""

with open('/content/README.md', 'w') as f:
    f.write(readme_content)

print("README updated successfully")

README updated successfully


In [35]:
import subprocess

result = subprocess.run(['find', '/drive/MyDrive', '-name', '*.ipynb'],
                      capture_output=True, text=True)
print(result.stdout)

/drive/MyDrive/Colab Notebooks/11-7-Confusion_Matrix.ipynb
/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/drive/MyDrive/Colab Notebooks/Research Paper M (1).ipynb
/drive/MyDrive/Colab Notebooks/formalize.ipynb
/drive/MyDrive/Colab Notebooks/VGG19 LUNGS_final.ipynb
/drive/MyDrive/Colab Notebooks/Lung-Master.ipynb
/drive/MyDrive/Colab Notebooks/formalize_backup.ipynb



In [39]:
import json

with open('/drive/MyDrive/Colab Notebooks/formalize.ipynb', 'r') as f:
    nb = json.load(f)

print(f"Total cells: {len(nb.get('cells', []))}")
print("Last 3 cell previews:")
for cell in nb['cells'][-3:]:
    source = ''.join(cell.get('source', []))
    print(f"  - {source[:80]}...")

Total cells: 19
Last 3 cell previews:
  - import json

notebook_path = '/drive/MyDrive/Colab Notebooks/formalize.ipynb'

#...
  - import json

# Find the notebook in Google Drive
from google.colab import drive
...
  - readme_content = """# Neuro-Symbolic Math Data Generation

Implementation of the...


In [42]:
import json

with open('/drive/MyDrive/Colab Notebooks/formalize.ipynb', 'r') as f:
    nb = json.load(f)

# Deep clean
if 'metadata' in nb:
    nb['metadata'].pop('widgets', None)
    nb['metadata'].pop('accelerator', None)

for cell in nb.get('cells', []):
    if 'metadata' in cell:
        cell['metadata'] = {}
    if 'outputs' in cell:
        clean_outputs = []
        for output in cell['outputs']:
            if output.get('output_type') != 'display_data':
                clean_outputs.append(output)
        cell['outputs'] = clean_outputs

print(f"Total cells: {len(nb.get('cells', []))}")

with open('/content/pipeline_clean.ipynb', 'w') as f:
    json.dump(nb, f, indent=2)

print("Done - download pipeline_clean.ipynb")

Total cells: 19
Done - download pipeline_clean.ipynb


In [43]:
import json

notebook_path = '/drive/MyDrive/Colab Notebooks/formalize.ipynb'

# Read the notebook
with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Fix the metadata widgets issue
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']
    print("Removed widgets metadata")
else:
    print("No widgets metadata found")

# Save fixed version
fixed_path = '/content/pipeline.ipynb'
with open(fixed_path, 'w') as f:
    json.dump(nb, f, indent=2)

print("Fixed notebook saved as pipeline.ipynb")


Removed widgets metadata
Fixed notebook saved as pipeline.ipynb


In [44]:
import json

# Find the notebook in Google Drive
from google.colab import drive
drive.mount('/drive')

import os
result = os.popen("find /drive -name '*.ipynb' 2>/dev/null").read()
print(result)

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).
/drive/MyDrive/Colab Notebooks/11-7-Confusion_Matrix.ipynb
/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/drive/MyDrive/Colab Notebooks/Research Paper M (1).ipynb
/drive/MyDrive/Colab Notebooks/formalize.ipynb
/drive/MyDrive/Colab Notebooks/VGG19 LUNGS_final.ipynb
/drive/MyDrive/Colab Notebooks/Lung-Master.ipynb
/drive/MyDrive/Colab Notebooks/formalize_backup.ipynb

